In [ ]:
# Cell 0: 硬件自检
import torch
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"显存: {vram:.1f} GB")
else:
    print("⚠️ 没检测到 GPU。手算交叉熵的 cell（5.2）能在 CPU 跑；"
          "加载 Qwen3-8B 的 cell 需要 T4 及以上，请切到 GPU 运行时。")

In [ ]:
%%capture
# Cell 1: 安装依赖
# %%capture 必须是 cell 第一行，把冗长的 pip 日志藏起来。
# transformers>=4.51：Qwen3 架构支持；accelerate：device_map 自动放置；
# bitsandbytes：4-bit NF4 量化，让 8B 模型在 T4（15 GB）上放得下。
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

In [ ]:
# Cell 2: 手算交叉熵 vs F.cross_entropy（纯 CPU 小张量）
import torch
import torch.nn.functional as F

# 假设词表只有 5 个 token，模型在某个位置输出这条 raw logits：
logits = torch.tensor([2.0, 1.0, 0.1, -1.0, 0.5])   # [V=5]
label  = torch.tensor(1)                             # 真实的下一个 token 是 id=1

# —— 手算：softmax → 取真实类别的概率 → 负对数 ——
probs = torch.softmax(logits, dim=-1)               # 先归一化成概率分布
p_true = probs[label]                               # 真实 token 那一项的概率
manual_loss = -torch.log(p_true)                    # ℓ = -log q[label]
print(f"softmax 概率: {probs.numpy().round(4)}")
print(f"真实 token 概率 q[1] = {p_true:.4f}")
print(f"手算 loss = {manual_loss:.4f}")

# —— F.cross_entropy：直接吃 raw logits（注意要加 batch 维）——
auto_loss = F.cross_entropy(logits.unsqueeze(0), label.unsqueeze(0))
print(f"F.cross_entropy = {auto_loss:.4f}")         # 与手算一致
assert torch.allclose(manual_loss, auto_loss), "应当相等"


In [ ]:
# Cell 3: 4-bit 量化加载 Qwen3-8B
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen3-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # 开 4-bit 量化
    bnb_4bit_quant_type="nf4",               # NF4：对正态分布权重更友好的 4-bit 类型
    bnb_4bit_use_double_quant=True,          # 双量化，再省一点显存
    bnb_4bit_compute_dtype=torch.float16,    # T4 是 Turing，不支持 bf16，用 fp16 计算
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto",
)
model.eval()                                 # 推理 / 算 loss 都不需要 dropout
print("加载完成，参数量（含量化权重）:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Cell 4: 真实文本上的自回归 loss 与 perplexity
import torch, torch.nn.functional as F

text = "The capital of France is Paris."
ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)  # [1, L]
print("token 数 L =", ids.shape[1])

with torch.no_grad():
    logits = model(ids).logits          # [1, L, V]，每个位置一条「下一个 token」打分

# —— 手动 shift：错开一位对齐（2.3 节）——
shift_logits = logits[:, :-1, :]        # [1, L-1, V] 丢掉最后一个位置（无标签）
shift_labels = ids[:, 1:]               # [1, L-1]   丢掉第一个 token（不作标签）

# 每个位置一条 ℓ_t，先看清「逐位置」的 loss
per_token = F.cross_entropy(
    shift_logits.reshape(-1, shift_logits.size(-1)),  # [L-1, V]
    shift_labels.reshape(-1),                         # [L-1]
    reduction="none",                                 # 不平均，保留每个位置
)
toks = tokenizer.convert_ids_to_tokens(shift_labels[0])  # 每个位置要预测的真实下一个 token
for tok, l in zip(toks, per_token):
    print(f"  预测下一个 token {tok!r:>12} 的 loss = {l.item():.3f}")  # loss 越小=模型越笃定

mean_loss = per_token.mean()
ppl = torch.exp(mean_loss)
print(f"\n平均 cross-entropy loss = {mean_loss:.4f}")
print(f"perplexity = exp(loss) = {ppl:.2f}")

# —— 和模型内置的自动 shift 对齐（2.3 节那个 labels=input_ids 便利）——
with torch.no_grad():
    builtin = model(ids, labels=ids).loss
print(f"模型内置 loss（labels=input_ids，自动 shift）= {builtin:.4f}")  # 与手算一致

In [ ]:
# Cell 5: teacher forcing 的并行 == 逐前缀串行
import torch

ids = tokenizer("Machine learning is fun", return_tensors="pt").input_ids.to(model.device)
L = ids.shape[1]

with torch.no_grad():
    # ① 一次并行：整句喂进去，拿到所有位置的 logits
    parallel_logits = model(ids).logits[0]          # [L, V]

    # ② 逐前缀串行：对每个前缀 ids[:, :t+1] 单独前向，取它最后一个位置的 logits
    serial_last = []
    for t in range(L):
        prefix = ids[:, : t + 1]                    # 只喂前 t+1 个 token
        logits_t = model(prefix).logits[0, -1]      # 该前缀最后位置 = 预测第 t+1 个 token
        serial_last.append(logits_t)
    serial_logits = torch.stack(serial_last)        # [L, V]

# 逐位置对比：并行的第 t 行 应当 == 串行喂前缀 ids[:, :t+1] 的最后一行
max_diff = (parallel_logits - serial_logits).abs().max()
print(f"并行 vs 串行 logits 最大差异 = {max_diff:.2e}")   # 量级远小于 logits 本身（4-bit 反量化+fp16 累积误差，通常 1e-3~1e-1）
print("结论：因果掩码下，一次并行前向 = 逐前缀分别前向——位置 t 只依赖它的前文。")

In [ ]:
# Cell 6: 手写贪心自回归生成（推理 = 从训练学到的分布里一步步采样）
import torch

prompt = "The capital of France is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

generated = ids
with torch.no_grad():
    for step in range(5):                          # 生成 5 个 token
        logits = model(generated).logits           # [1, cur_len, V]
        next_logits = logits[0, -1]                # ★只取最后一个位置（4.2 节）
        next_id = next_logits.argmax()             # greedy = 取概率最大者（4.3 节）
        # 顺手看一眼这一步分布有多「笃定」：最大概率是多少
        p = torch.softmax(next_logits, dim=-1)[next_id]
        tok = tokenizer.decode(next_id)
        print(f"step {step}: 选中 {tok!r}（p={p:.3f}）")
        generated = torch.cat([generated, next_id.view(1, 1)], dim=1)  # 拼回前文 → 下一轮

print("\n完整输出:", tokenizer.decode(generated[0]))

# —— exposure bias 的直观点题 ——
# 训练时位置 t 的前文是【真实语料】；推理时（上面循环）前文是【模型自己生成的 token】。
# 一旦某步采样出一个略差的 token，它就进入后续所有步的前文——而这种「带瑕疵的前文」
# 在 teacher-forcing 训练里几乎没出现过（3.4 节）。greedy 因为每步都挑最大，
# 还容易陷入重复；第 2 章的 temperature / top-p 正是在这个分布上做文章来缓解。